# Predicting Residential Property Sale Prices in New York City (2025)
**Module:** 6COSC017C Machine Learning and Data Analytics — CW1

**Business question:** Can we predict the sale price of a residential property in New York City from its physical characteristics, location and sale timing — and which factors drive price the most?

**Dataset:** NYC Department of Finance — Annualized Property Sales, calendar year 2025 (five borough files).
Source: https://www.nyc.gov/site/finance/property/property-annualized-sales-update.page (public NYC government data).

**Notebook structure**
1. Load data (combine 5 borough files)
2. First look at the data
3. Exploratory Data Analysis *(next)*
4. Data preparation *(next)*
5. Model training *(next)*
6. Model evaluation *(next)*

## 0. Setup — libraries

In [1]:
# Core data handling
import pandas as pd
import numpy as np

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings so wide tables are readable
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
sns.set_theme(style="whitegrid")

# Fixed random seed so every result in this notebook is reproducible
RANDOM_STATE = 42

## 1. Load data

The Department of Finance publishes one Excel file per borough. Each file has **6 title/note lines** at the top, then the column headers on line 7, followed by one empty row.
We therefore:
1. read each file with `header=6` (0-based row index of the header line),
2. drop fully empty rows,
3. add a `borough_name` column so we keep track of where each row came from,
4. stack (concatenate) the five files into one DataFrame — they share exactly the same 21 columns, so a vertical concatenation is the correct way to combine them.

In [2]:
# File name -> readable borough name
BOROUGH_FILES = {
    "2025_manhattan.xlsx":     "Manhattan",
    "2025_bronx.xlsx":         "Bronx",
    "2025_brooklyn.xlsx":      "Brooklyn",
    "2025_queens.xlsx":        "Queens",
    "2025_staten_island.xlsx": "Staten Island",
}

frames = []
for file_name, borough in BOROUGH_FILES.items():
    df_b = pd.read_excel(file_name, header=6)   # header is on the 7th line
    df_b = df_b.dropna(how="all")               # remove empty spacer rows
    df_b["borough_name"] = borough              # remember the source file
    frames.append(df_b)
    print(f"{borough:<14} {len(df_b):>7,} rows")

df_raw = pd.concat(frames, ignore_index=True)
print(f"\nCombined dataset: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")

Manhattan       19,724 rows


Bronx            6,818 rows


Brooklyn        23,472 rows


Queens          27,258 rows


Staten Island    7,796 rows

Combined dataset: 85,068 rows x 22 columns


### 1.1 Clean column names
The original headers contain line breaks, double spaces and capital letters (e.g. `'GROSS \nSQUARE FEET'`). We convert them to `snake_case` so they are easy to use in code.

In [3]:
def clean_col(name: str) -> str:
    name = " ".join(str(name).split())          # remove line breaks / extra spaces
    return name.lower().replace("-", "").replace(" ", "_")

df_raw.columns = [clean_col(c) for c in df_raw.columns]
df_raw.columns.tolist()

['borough',
 'neighborhood',
 'building_class_category',
 'tax_class_at_present',
 'block',
 'lot',
 'easement',
 'building_class_at_present',
 'address',
 'apartment_number',
 'zip_code',
 'residential_units',
 'commercial_units',
 'total_units',
 'land_square_feet',
 'gross_square_feet',
 'year_built',
 'tax_class_at_time_of_sale',
 'building_class_at_time_of_sale',
 'sale_price',
 'sale_date',
 'borough_name']

### 1.2 Sanity checks after loading
We confirm that every borough loaded, that the sale dates cover the full 2025 calendar year, and that the combination produced no duplicate rows.

In [4]:
print("Rows per borough:")
print(df_raw["borough_name"].value_counts(), "\n")

print("Sale date range:", df_raw["sale_date"].min().date(), "to", df_raw["sale_date"].max().date())
print("Exact duplicate rows:", df_raw.duplicated().sum())

Rows per borough:
borough_name
Queens           27258
Brooklyn         23472
Manhattan        19724
Staten Island     7796
Bronx             6818
Name: count, dtype: int64 

Sale date range: 2025-01-01 to 2025-12-31
Exact duplicate rows: 0


## 2. First look at the data

In [5]:
df_raw.head()

,borough,neighborhood,building_class_category,tax_class_at_present,block,lot,easement,building_class_at_present,address,apartment_number,zip_code,residential_units,commercial_units,total_units,land_square_feet,gross_square_feet,year_built,tax_class_at_time_of_sale,building_class_at_time_of_sale,sale_price,sale_date,borough_name
0,1.00,ALPHABET CITY,01 ONE FAMILY DWELLINGS,1,376.00,43.00,NaN,S1,743 EAST 6 STREET,NaN,"10,009.00",1.00,1.00,2.00,"2,090.00","3,680.00","1,940.00",1.00,S1,0.00,2025-01-23,Manhattan
1,1.00,ALPHABET CITY,07 RENTALS - WALKUP APARTMENTS,2B,372.00,9.00,NaN,C7,272 EAST 3 STREET,NaN,"10,009.00",8.00,1.00,9.00,"2,021.00","7,129.00","1,900.00",2.00,C7,0.00,2025-12-15,Manhattan
2,1.00,ALPHABET CITY,07 RENTALS - WALKUP APARTMENTS,2B,372.00,19.00,NaN,C7,"292 EAST 3 STREET, 4B",NaN,"10,009.00",9.00,1.00,10.00,"2,401.00","6,920.00","1,920.00",2.00,C7,0.00,2025-02-07,Manhattan
3,1.00,ALPHABET CITY,07 RENTALS - WALKUP APARTMENTS,2,376.00,41.00,NaN,C6,"745 EAST 6TH STREET, 1B",NaN,"10,009.00",28.00,0.00,28.00,"4,542.00","18,254.00","1,900.00",2.00,C4,"540,000.00",2025-12-17,Manhattan
4,1.00,ALPHABET CITY,07 RENTALS - WALKUP APARTMENTS,2,385.00,38.00,NaN,C6,"21-23 AVENUE C, 5B",NaN,"10,009.00",20.00,2.00,22.00,"3,315.00","17,160.00","1,900.00",2.00,C4,0.00,2025-11-11,Manhattan


### 2.1 Data types
`info()` shows each column's type and how many non-missing values it has. This tells us which columns are numeric (usable directly in models) and which are text/categorical (need encoding later).

In [6]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 85068 entries, 0 to 85067
Data columns (total 22 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   borough                         85068 non-null  float64       
 1   neighborhood                    85068 non-null  object        
 2   building_class_category         85068 non-null  object        
 3   tax_class_at_present            85068 non-null  object        
 4   block                           85068 non-null  float64       
 5   lot                             85068 non-null  float64       
 6   easement                        0 non-null      float64       
 7   building_class_at_present       85068 non-null  object        
 8   address                         85068 non-null  object        
 9   apartment_number                20679 non-null  object        
 10  zip_code                        85049 non-null  float64       
 11  re

### 2.2 Missing values
Missing values must be identified before modelling, because most scikit-learn models cannot handle them. We show the count and percentage per column.

In [7]:
missing = pd.DataFrame({
    "missing_count": df_raw.isna().sum(),
    "missing_pct":   (df_raw.isna().mean() * 100).round(1),
}).sort_values("missing_pct", ascending=False)
missing[missing["missing_count"] > 0]

,missing_count,missing_pct
easement,85068,100.00
apartment_number,64389,75.70
gross_square_feet,38358,45.10
land_square_feet,38360,45.10
commercial_units,35207,41.40
residential_units,20663,24.30
total_units,17510,20.60
year_built,5562,6.50
zip_code,19,0.00


**Observations**
- `easement` is 100% empty → it carries no information and will be dropped.
- `apartment_number` is mostly empty because it only applies to individual apartment units.
- `land_square_feet` and `gross_square_feet` are missing for many rows. In this dataset, co-op and condo sales are recorded **per apartment unit**, so building size is not recorded for them. This drives our scope decision in the Data Preparation section.

### 2.3 The target variable: `sale_price`
Not every record is a real market sale. Transfers between family members, or deeds transferred for legal reasons, are recorded with a price of **$0** or a token amount. These are not market prices and would mislead a price model.

In [8]:
price = df_raw["sale_price"]
print(f"Sales recorded at $0:        {(price == 0).sum():>7,}")
print(f"Sales below $10,000:         {(price < 10_000).sum():>7,}")
print(f"Sales of $10,000 or more:    {(price >= 10_000).sum():>7,}")
print()
price[price >= 10_000].describe()

Sales recorded at $0:         29,608
Sales below $10,000:          31,436
Sales of $10,000 or more:     53,632



count          53,632.00
mean        3,265,885.50
std        18,620,058.12
min            10,000.00
25%           560,000.00
50%           885,000.00
75%         1,550,000.00
max     1,080,000,000.00
Name: sale_price, dtype: float64

**Observation:** among real sales, the median price is far below the mean and the maximum is over $1 billion → the target is **heavily right-skewed**. We will look at this in the EDA and consider a log transformation.

### 2.4 Building class categories
The dataset mixes very different property types (houses, apartments, offices, garages, vacant land). Predicting one price for all of them would not be meaningful, so we look at what is there.

In [9]:
df_raw["building_class_category"].value_counts().head(15)

building_class_category
01 ONE FAMILY DWELLINGS              18578
02 TWO FAMILY DWELLINGS              15732
10 COOPS - ELEVATOR APARTMENTS       13620
13 CONDOS - ELEVATOR APARTMENTS      13464
03 THREE FAMILY DWELLINGS             4522
07 RENTALS - WALKUP APARTMENTS        2880
09 COOPS - WALKUP APARTMENTS          2448
04 TAX CLASS 1 CONDOS                 1620
15 CONDOS - 2-10 UNIT RESIDENTIAL     1507
44 CONDO PARKING                      1434
17 CONDO COOPS                        1304
12 CONDOS - WALKUP APARTMENTS         1000
05 TAX CLASS 1 VACANT LAND             965
22 STORE BUILDINGS                     825
29 COMMERCIAL GARAGES                  598
Name: count, dtype: int64

**Observation:** the three most common categories are 1-family, 2-family and 3-family homes. For these, building size, land size and year built are recorded, which makes them the best-suited group for a price-prediction model.

---
*Next: Section 3 — Exploratory Data Analysis on the residential (1–3 family) subset.*